# Uber Fare Amount Prediction - Data Preprocessing & Exploratory Data Analysis

## Objective
The primary objective of this project is to **predict the fare amount** for Uber rides using **regression analysis**.

This notebook focuses on the **Data Preprocessing and Exploratory Data Analysis (EDA)** phase, a critical step before developing any machine learning model.

---

## Scope of This Notebook

In this phase, we will carry out the following steps:

1. **Data Cleaning**
   - Handling missing datas
   - Converting data types, if necessary

2. **Exploratory Data Analysis (EDA)**
   - Understanding data distribution and trends
   - Visualizing key variables and relationships

3. **Outlier Detection & Treatment**
   - Identifying and handling outliers that may skew the model

4. **Feature Engineering**
   - Creating meaningful new features from existing data
   - Transforming variables for better model performance

5. **Feature Importance Analysis**
   - Identifying the most relevant features using model-based techniques

---

## Next Steps
Based on the insights and features derived in this notebook, we will proceed to build, evaluate, and fine-tune machine learning models for fare amount prediction.

---

_This work is part of my internship project at **Uber**, where the aim is to develop robust and interpretable ML solutions for real-world pricing systems._

In [2]:
#imports
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from numpy.testing.print_coercion_tables import print_new_cast_table
%matplotlib inline

In [3]:
df = pd.read_csv('uber.csv')
df.head()

,Unnamed: 0,key,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
0,24238194,2015-05-07 19:52:06.0000003,7.5,2015-05-07 19:52:06 UTC,-73.999817,40.738354,-73.999512,40.723217,1
1,27835199,2009-07-17 20:04:56.0000002,7.7,2009-07-17 20:04:56 UTC,-73.994355,40.728225,-73.994710,40.750325,1
2,44984355,2009-08-24 21:45:00.00000061,12.9,2009-08-24 21:45:00 UTC,-74.005043,40.740770,-73.962565,40.772647,1
3,25894730,2009-06-26 08:22:21.0000001,5.3,2009-06-26 08:22:21 UTC,-73.976124,40.790844,-73.965316,40.803349,3
4,17610152,2014-08-28 17:47:00.000000188,16.0,2014-08-28 17:47:00 UTC,-73.925023,40.744085,-73.973082,40.761247,5


In [4]:
df.shape

(200000, 9)

In [5]:
df.columns

Index(['Unnamed: 0', 'key', 'fare_amount', 'pickup_datetime',
       'pickup_longitude', 'pickup_latitude', 'dropoff_longitude',
       'dropoff_latitude', 'passenger_count'],
      dtype='object')

**Here we will drop the columns 'Unnamed: 0' and 'key' because the key column is just the duplicate column for pickup_datetime(column) and key is likely an index for each rows and they doesn't contribute that much to our analysis. so, dropping them will be a better choice.**

In [6]:
df = df.drop(['Unnamed: 0', 'key'], axis=1)

In [7]:
#lets see the head of the dataset again
df.head()

,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
0,7.5,2015-05-07 19:52:06 UTC,-73.999817,40.738354,-73.999512,40.723217,1
1,7.7,2009-07-17 20:04:56 UTC,-73.994355,40.728225,-73.994710,40.750325,1
2,12.9,2009-08-24 21:45:00 UTC,-74.005043,40.740770,-73.962565,40.772647,1
3,5.3,2009-06-26 08:22:21 UTC,-73.976124,40.790844,-73.965316,40.803349,3
4,16.0,2014-08-28 17:47:00 UTC,-73.925023,40.744085,-73.973082,40.761247,5


In [8]:
df.dtypes

,0
fare_amount,float64
pickup_datetime,object
pickup_longitude,float64
pickup_latitude,float64
dropoff_longitude,float64
dropoff_latitude,float64
passenger_count,int64


lets do data validation and we see that the column pickup date time is an object dtype so lets convert this column into datetime format so that in the futuew we can do better feature engineering process and use advance techniques to extract information like date day of the week etc.

In [9]:
df["pickup_datetime"] = pd.to_datetime(df["pickup_datetime"])

In [10]:
df["pickup_datetime"].dtypes

datetime64[ns, UTC]

**we successfully converted the column into datetime dtype now to the next phase we will handle outliers in each column and clean the missing values**

# HANDELING OUTLIERS

## Outlier Handling Strategy

In this project, a traditional outlier handling approach (like capping or imputing extreme values) is not appropriate due to the nature of the dataset and the importance of preserving data integrity.

Here’s the reasoning by column:

- **`fare_amount`**
  This is the target variable. No imputation is performed. Instead, we may remove records with clearly invalid values (e.g., negative fares or extreme values inconsistent with real-world trips).

- **`pickup_datetime`**
  Timestamps are not treated as numerical variables for outlier detection. Instead, we check for logically invalid entries (e.g., future years or malformed dates) and extract relevant time-based features (hour, weekday, etc.).

- **`pickup_latitude`, `pickup_longitude`, `dropoff_latitude`, `dropoff_longitude`**
  These represent geospatial coordinates. Imputing or modifying them would compromise the validity of the data.
  Outlier handling is done through **geographic bounding**, where rows with coordinates outside a reasonable New York City range are removed:
  - Latitude: [-180, 180]
  - Longitude: [-90, 90]

- **`passenger_count`**
  Only logical passenger counts (e.g., 1–6) are retained. Rows with values like 0, negative numbers, or implausible high values are removed rather than imputed.

---

### ✅ Summary

This is a real-world transportation dataset where **context matters more than pure statistical thresholds**.
Our approach ensures data quality without introducing artificial bias or violating domain integrity. A crucial step before building a reliable machine learning model.


In [11]:
df.columns

Index(['fare_amount', 'pickup_datetime', 'pickup_longitude', 'pickup_latitude',
       'dropoff_longitude', 'dropoff_latitude', 'passenger_count'],
      dtype='object')

In [12]:
# cleaning fare_amount column and removing negative values.
df = df[df["fare_amount"] > 0]

In [13]:
# lets validate the pickup date time column and see if the date is in the range or not...
df[df["pickup_datetime"] >= '2015']

,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
0,7.5,2015-05-07 19:52:06+00:00,-73.999817,40.738354,-73.999512,40.723217,1
10,6.5,2015-05-22 17:32:27+00:00,-73.974388,40.746952,-73.988586,40.729805,1
18,12.0,2015-03-25 08:58:35+00:00,-73.962532,40.767189,-73.974457,40.753860,1
21,5.0,2015-03-03 23:15:03+00:00,-73.989189,40.729141,-73.987282,40.720634,2
47,12.0,2015-01-04 09:17:47+00:00,-73.979523,40.727310,-73.984879,40.760651,1
...,...,...,...,...,...,...,...
199984,9.0,2015-06-08 12:34:33+00:00,-73.988243,40.759258,-73.972778,40.755070,1
199985,24.0,2015-04-18 15:16:06+00:00,-74.005089,40.737301,-73.945290,40.774162,5
199990,12.0,2015-05-24 22:05:56+00:00,-73.987106,40.741894,-73.952240,40.772957,1
199991,17.5,2015-06-08 10:49:14+00:00,-73.981453,40.743919,-74.013908,40.712635,1


In [14]:
out_of_bounds_latlong = df[~(
    (df['pickup_latitude'].between(-90, 90)) &
    (df['dropoff_latitude'].between(-90, 90)) &
    (df['pickup_longitude'].between(-180, 180)) &
    (df['dropoff_longitude'].between(-180, 180))
)]

out_of_bounds_latlong

,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
4949,4.9,2012-04-28 00:58:00+00:00,-748.016667,40.739957,-74.003570,40.734192,1
32549,15.7,2012-06-16 10:04:00+00:00,-74.016055,40.715155,-737.916665,40.697862,2
48506,33.7,2011-11-05 23:26:00+00:00,-735.200000,40.770092,-73.980187,40.765530,1
56617,8.1,2012-03-11 07:24:00+00:00,-73.960828,404.433332,-73.988357,40.769037,1
61793,8.5,2012-06-13 05:45:00+00:00,-73.951385,401.066667,-73.982110,40.754117,1
75851,15.7,2011-11-05 00:22:00+00:00,-1340.648410,1644.421482,-3356.666300,872.697628,1
87946,24.1,2013-07-02 03:51:57+00:00,-73.950581,40.779692,NaN,NaN,0
91422,16.1,2011-05-18 13:24:00+00:00,57.418457,1292.016128,1153.572603,-881.985513,1
103745,12.9,2011-10-14 19:04:00+00:00,-736.216667,40.767035,-73.982377,40.725562,1
139447,13.7,2012-01-20 11:50:00+00:00,-74.011042,40.709780,-73.983163,493.533332,4


here we can see there are 13 observations that are out of the lat long boundry which it might be an error in data recording

In [15]:
#lets fix this and only keep the observations that are in the lat long boundry i.e latitude [-180, 180] and longitude [-90, 90].
df = df[
    (df['pickup_latitude'].between(-90, 90)) &
    (df['dropoff_latitude'].between(-90, 90)) &
    (df['pickup_longitude'].between(-180, 180)) &
    (df['dropoff_longitude'].between(-180, 180))
]


In [16]:
# lets clean the passenger count column.
df[(df["passenger_count"] < 0) | (df["passenger_count"] > 6)]

,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
113038,11.7,2010-12-28 08:20:00+00:00,-73.937795,40.758498,-73.937835,40.758415,208


we can see there is no anamoly in the passenger count column as they have perfect values in the range no less then 0 and no grater then 6 as we the data needs to be.

# lets handle the missing data

### Handling Missing Values: Best Practices

To handle missing values, we follow a simple rule:

1. **Less than 5% missing values**:
   - If the percentage of missing values in a feature is less than 5%, it is generally safe to drop the missing values. This approach ensures that the data integrity remains intact, as the removal of a small number of data points typically does not affect the overall distribution of the dataset. Additionally, the performance of future models is unlikely to be negatively impacted by such drops, making this a practical solution for managing missing data.

2. **More than 5% missing values**:
    - If the missing values in a feature exceed 5%, dropping them could result in a significant loss of data, which may impact the model's ability to generalize well. In such cases, imputation is a more appropriate strategy. Depending on the nature of the feature.


This approach helps maintain data quality and ensures optimal model performance.


In [17]:
# lets see what is the 5% of data in numbers which is a threshhold for us to drop the missing values.
threshhold = (df.shape[0] * 0.05)
print(f"total amount of data we can drop: {threshhold}")
# we can drop 9998 amount of total data which will not lets check the shape of our data.
print(f"total data we have: {df.shape[0]}")
final_data_thres = df.shape[0] - threshhold
print(f"the total data we need after decucting the 5%: {final_data_thres}")

total amount of data we can drop: 9998.25
total data we have: 199965
the total data we need after decucting the 5%: 189966.75


In [18]:
df.isna().sum()

,0
fare_amount,0
pickup_datetime,0
pickup_longitude,0
pickup_latitude,0
dropoff_longitude,0
dropoff_latitude,0
passenger_count,0


In [19]:
for cols in df.columns:
    print(f"{cols}: {df[df[cols] == 0][cols].count()}")

fare_amount: 0
pickup_datetime: 0
pickup_longitude: 3785
pickup_latitude: 3781
dropoff_longitude: 3761
dropoff_latitude: 3755
passenger_count: 708


In [20]:
for col in df.columns:
    df.drop(df[df[col] == 0].index, inplace=True)

In [21]:
if df.shape[0] <= final_data_thres:
    print("ah ah this is not good!!")
else:
    print("this is good!!!")

this is good!!!


# What’s Happening??

We began by investigating our dataset to identify any suspicious or invalid data entries. It quickly became clear that several columns had values of `0` particularly in fields like `pickup_longitude`, `dropoff_latitude`, and `passenger_count`. These zeros aren’t just unusual they’re **invalid** in the context of a taxi ride dataset. A trip cannot happen without valid coordinates or passengers.

---

## Cleaning It Up

To ensure data quality, we decided to remove all rows that had a `0` in any of these critical columns. This was a strict data-cleaning step removing entire rows where key features contained invalid values. The idea was to eliminate any records that could introduce noise or bias into our analysis or model training.

---

## Following Best Practices

We didn’t just delete blindly — we followed a smart rule:

    > If less than 5% of the data is invalid or missing, it's safe to drop.
    > If more than 5% is affected, dropping might hurt the model, so consider other strategies like imputation.

We calculated the threshold for 5% of the dataset and compared it with how much data we actually dropped. Fortunately, the number of dropped records was **well within the 5% limit**, so our aggressive cleanup was justified.

---

## The Verdict

Once we cleaned the data, we double-checked our new dataset size and confirmed:

**“this is good!!!”**

We’re now left with a cleaner, more reliable dataset ready for solid analysis and modeling.


# Lets do Feature engineering

In [22]:
#lets calculate the pickup to dropoff distance we will use the haversine formula. we will import a library for a clean approach which Handles the math under the hood, also we can easily switch between miles, kilometers, nautical miles, etc.
!pip install haversine
from haversine import haversine, Unit

# creating a function to apply row-wise.
def calculate_distance(row):
    pickup = (row['pickup_latitude'], row['pickup_longitude'])
    dropoff = (row['dropoff_latitude'], row['dropoff_longitude'])
    return haversine(pickup, dropoff, unit=Unit.KILOMETERS)

# calling the function using the pandas apply method.
df['distance_km'] = df.apply(calculate_distance, axis=1)

In [23]:
# datetime features extraction lets extract what we can hour, day, month, year, day of week, is weekend etc..
df['hour'] = df['pickup_datetime'].dt.hour
df['day'] = df['pickup_datetime'].dt.day
df['month'] = df['pickup_datetime'].dt.month
df['year'] = df['pickup_datetime'].dt.year
df['day_of_week'] = df['pickup_datetime'].dt.dayofweek
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

# fare per km calculation
df['fare_per_km'] = df['fare_amount'] / (df['distance_km'] + 1e-3) # adding small constant(0.001) to avoid division by 0, "if there is still any", cant take chances.

In [24]:
# lets see the first five rows of the dataset and total columns of the dataset
df.head()

,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count,distance_km,hour,day,month,year,day_of_week,is_weekend,fare_per_km
0,7.5,2015-05-07 19:52:06+00:00,-73.999817,40.738354,-73.999512,40.723217,1,1.683325,19,7,5,2015,3,0,4.452822
1,7.7,2009-07-17 20:04:56+00:00,-73.994355,40.728225,-73.994710,40.750325,1,2.457593,20,17,7,2009,4,0,3.131872
2,12.9,2009-08-24 21:45:00+00:00,-74.005043,40.740770,-73.962565,40.772647,1,5.036384,21,24,8,2009,0,0,2.560853
3,5.3,2009-06-26 08:22:21+00:00,-73.976124,40.790844,-73.965316,40.803349,3,1.661686,8,26,6,2009,4,0,3.187614
4,16.0,2014-08-28 17:47:00+00:00,-73.925023,40.744085,-73.973082,40.761247,5,4.475456,17,28,8,2014,3,0,3.574256


In [25]:
df.columns

Index(['fare_amount', 'pickup_datetime', 'pickup_longitude', 'pickup_latitude',
       'dropoff_longitude', 'dropoff_latitude', 'passenger_count',
       'distance_km', 'hour', 'day', 'month', 'year', 'day_of_week',
       'is_weekend', 'fare_per_km'],
      dtype='object')

In [26]:
df.shape

(195314, 15)

# lets do correlation analysis and see how each feature is correlated with the target feature "fare_amount"

In [27]:
df.corr(numeric_only=True)

,fare_amount,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count,distance_km,hour,day,month,year,day_of_week,is_weekend,fare_per_km
fare_amount,1.000000,0.010974,-0.008027,0.009414,-0.008065,0.010007,0.036202,-0.020906,0.000675,0.023844,0.118675,0.006379,-0.000060,0.196642
pickup_longitude,0.010974,1.000000,-0.965028,0.952395,-0.961019,0.009258,0.145750,0.002318,0.019125,-0.008022,0.013306,0.008010,0.001163,0.003918
pickup_latitude,-0.008027,-0.965028,1.000000,-0.966219,0.993183,-0.008661,-0.021031,-0.001822,-0.019965,0.008190,-0.014613,-0.008592,-0.001740,0.000339
dropoff_longitude,0.009414,0.952395,-0.966219,1.000000,-0.966602,0.009163,0.131015,0.001649,0.019002,-0.007182,0.012656,0.008070,0.001908,0.003919
dropoff_latitude,-0.008065,-0.961019,0.993183,-0.966602,1.000000,-0.009398,-0.036436,-0.001843,-0.020004,0.007815,-0.014245,-0.008650,-0.001638,0.000353
passenger_count,0.010007,0.009258,-0.008661,0.009163,-0.009398,1.000000,0.004361,0.013187,0.003648,0.009474,0.004816,0.033107,0.039171,-0.006387
distance_km,0.036202,0.145750,-0.021031,0.131015,-0.036436,0.004361,1.000000,-0.001198,0.000118,-0.001063,-0.000107,0.001985,0.002612,-0.003728
hour,-0.020906,0.002318,-0.001822,0.001649,-0.001843,0.013187,-0.001198,1.000000,0.005166,-0.004291,0.001947,-0.086217,-0.090719,-0.004243
day,0.000675,0.019125,-0.019965,0.019002,-0.020004,0.003648,0.000118,0.005166,1.000000,-0.017327,-0.012036,0.005544,0.003711,0.003470
month,0.023844,-0.008022,0.008190,-0.007182,0.007815,0.009474,-0.001063,-0.004291,-0.017327,1.000000,-0.115385,-0.009466,-0.006521,0.003543


From the correlation matrix, we observe that most independent variables have a very weak correlation with the target variable `fare_amount`. This likely indicates that the relationships are **non-linear**, which is common in real-world datasets.

Pearson correlation only measures **linear associations**, so it may fail to capture more complex interactions between features and the target. As a result, the low correlations we see do not necessarily imply the features are unimportant.

To better understand feature contributions, we will use **Random Forest**, an ensemble learning technique based on decision trees. Random Forest can model non-linear relationships and provides feature importance metrics without requiring feature scaling.

Additionally, we will apply **XGBoost** an extreme gradient boosing tree-based method known for its performance in structured data problems, to validate our findings and further assess feature importance.


In [28]:
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

def get_feature_importances(model_type, X, y):
    if model_type == 'rf':
        model = RandomForestRegressor(n_estimators=100, random_state=42)
    elif model_type == 'xgb':
        model = XGBRegressor(n_estimators=100, random_state=42)
    else:
        raise ValueError("model_type must be 'rf' or 'xgb'")

    model.fit(X, y)
    importances = model.feature_importances_
    feature_names = X.columns

    return pd.DataFrame({'Feature': feature_names, 'Importance': importances}).sort_values(by='Importance', ascending=False)

In [29]:
X_f = df.drop(columns=['fare_amount', 'pickup_datetime'])
y_f = df['fare_amount']
get_feature_importances(model_type='rf', X=X_f, y=y_f)

,Feature,Importance
5,distance_km,0.781340
12,fare_per_km,0.211891
2,dropoff_longitude,0.001521
0,pickup_longitude,0.001292
3,dropoff_latitude,0.000978
10,day_of_week,0.000582
1,pickup_latitude,0.000562
6,hour,0.000502
9,year,0.000462
8,month,0.000378


In [30]:
get_feature_importances(model_type='xgb', X=X_f, y = y_f)

,Feature,Importance
5,distance_km,0.671265
12,fare_per_km,0.141150
10,day_of_week,0.027389
2,dropoff_longitude,0.027389
9,year,0.025670
1,pickup_latitude,0.020954
6,hour,0.020064
0,pickup_longitude,0.018469
3,dropoff_latitude,0.013341
7,day,0.012743


# Feature Selection Summary for Fare Prediction Model

## Final Selected Features

| Feature         | Description                                    |
|----------------|------------------------------------------------|
| `distance_km`   | Calculated distance between pickup and dropoff |
| `fare_per_km`   | Fare divided by distance (efficiency metric)   |
| `day_of_week`   | Day of the week (0 = Monday, 6 = Sunday)       |
| `year`          | Year of the ride                               |
| `hour`          | Hour of the day when the ride began            |
| `fare_amount`   | Target variable: total fare of the ride      |

---

## Dropped Features and Justifications

| Feature              | Reason for Removal                                                |
|----------------------|-------------------------------------------------------------------|
| `pickup_latitude`    | Very low importance in both models; info already in `distance_km` |
| `pickup_longitude`   | Same as above                                                    |
| `dropoff_latitude`   | Same as above                                                    |
| `dropoff_longitude`  | Very low score; keeping it while dropping latitude breaks logic  |
| `month`              | Extremely low importance (≪ 0.01); adds no predictive value       |
| `day`                | Same as `month`; tiny impact on target variable                   |
| `passenger_count`    | Almost zero importance in RF model; irrelevant to fare calculation|

---

## Final Feature Selection Code

```python
# Keeping only the most impactful features
selected_df = df[[
    "distance_km",
    "fare_per_km",
    "day_of_week",
    "year",
    "hour",
    "fare_amount"
]].copy()


In [31]:
selected_df = df[["distance_km", "fare_per_km", "day_of_week", "year", "hour", "fare_amount"]].copy()

## Feature scaling
**To improve the performance and reliability of our regression model, we will scale the selected numerical features so they are on a similar range. This is important because, in regression problems, features with larger values can disproportionately influence the model's learning process, potentially leading to inaccurate predictions. Although some machine learning models like decision trees are less affected by feature scales, it is a good practice to standardize the data, especially when working with models that assume normally distributed inputs or rely on distance-based calculations. We will use standardization, which transforms each feature to have a mean of zero and a standard deviation of one. This ensures that all features contribute equally to the model's training and helps achieve better model performance and generalization. we will use StanderdScaler to Standerdize the data.**

In [37]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X = selected_df.iloc[:, :-1].values
y = selected_df.iloc[:, -1].values
# first we will split the data and then scale and normalize them cause if we first scale and normalize the data and split then there will be the risk of data leakage.
# we will use 80% of the data for training and 20% for testing..
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
#Scaling the data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"The shape(dimention) or (rows, columns) of the xtrain  data is {X_train.shape}")
print(f"The shape(dimention) or (rows, columns) of the xtest data is {X_test.shape}")

The shape(dimention) or (rows, columns) of the xtrain  data is (156251, 5)
The shape(dimention) or (rows, columns) of the xtest data is (39063, 5)


In [38]:
selected_df.to_csv("clean_data.csv");

---
**we will use this same dataset to our next component.**

# The end✨
## Author
### Ankit chimariya

----